# Initialize project environemt

In [ ]:
import sys; from pathlib import Path; sys.path.insert(0, str(Path.home() / "Github" / "the-bible-catalog"))

# Dependencies

In [ ]:
from config.settings import *

In [ ]:
from config._util_functions import *

In [ ]:
from config._common_libraries import *

In [ ]:
from config._common_functions import *

In [ ]:
from config._bronze_functions import *

# Parameter(s)

In [ ]:
p_env = get_env(ENVIRONMENT)
p_debug = DEBUG
p_job_name = "extract_esv_to_bronze"

print(f"Environment: {p_env}")
print(f"Debug mode: {p_debug}")
print(f"Job Name: {p_job_name}")

# Initialize Handler(s)

In [ ]:
ingestor = ESVBibleIngestion(api_key=EXTERNAL_SERVICES["esv_api_token"])

# Extract ESV Bible via API Request

In [ ]:
# Initialize variables
error_message = None
status = "SUCCESS"

# Run process
timer.start()
try:
    # Extract
    df = ingestor.get_full_bible_df()

    # Upsert to DuckDB
    upsert_to_motherduck(
        df=df,
        database_name=f"ext_{p_env}",
        schema="bronze",
        table_name="bible_catalog",
        key_columns=["translation", "book", "chapter", "verse_number"]
    )
except Exception as e:
    status = "FAILED"
    error_message = str(e)
finally:
    timer.end()

# Notify messenger

In [ ]:
messenger.send_workflow_notification(
    workflow_name=f"{p_job_name}-{p_env}",
    job_name=p_job_name,
    status=status,
    timer=timer,
    environment=p_env,
    debug=p_debug,
    custom_message=f"The `{p_job_name}` scheduled notebook procesed successfully. The entire process took approximately {timer.get_elapsed_time()}." if status == "SUCCESS" else f"{p_job_name} failed with error: {error_message}"
)